In [1]:
# /workspace/notebooks/inference_viewer.ipynb
import os
import pandas as pd

from modules.config import *
from modules.utils import *
from modules.metrics import *
from modules.experiment_set import *

In [2]:
experiment_set1 = select_set("set1")
ground_truth_set1 = get_ground_truth(replays=list(experiment_set1.test_replays))
inference_vanilla_set1 = get_inference(experiment_set1.model_vanilla(),
                                       replays=list(experiment_set1.test_replays),
                                       k=5)
inference_kbrs_set1 = get_inference(experiment_set1.model_kbrs(), 
                                    replays=list(experiment_set1.test_replays),
                                    k=5)

In [3]:
experiment_set2 = select_set("set2")
ground_truth_set2 = get_ground_truth(replays=list(experiment_set2.test_replays))
inference_vanilla_set2 = get_inference(experiment_set2.model_vanilla(),
                                       replays=list(experiment_set2.test_replays),
                                       k=5)
inference_kbrs_set2 = get_inference(experiment_set2.model_kbrs(), 
                                    replays=list(experiment_set2.test_replays),
                                    k=5)

[FAIL] /workspace/predictions/all_correct_win4_b16_kbrs_20250915_071540/model_020/6219.rep/all_correct.json -> JSONDecodeError: unexpected end of data: line 41318158 column 13 (char 626318175)


In [4]:
experiment_set3 = select_set("set3")
ground_truth_set3 = get_ground_truth(replays=list(experiment_set3.test_replays))
inference_legacy_model_set1 = get_inference(experiment_set3.model_legacy(),
                                       replays=list(experiment_set3.test_replays),
                                       k=1)
inference_kbrs_set3 = get_inference(experiment_set3.model_kbrs(), 
                                    replays=list(experiment_set3.test_replays),
                                    k=5)

In [5]:
def get_inference_result(
        model_path: Path,
        ground_truth,
        label_method="all_correct",
        replays=None,
        max_workers=16,
        force=None,
        k=1
    ):
        prediction = read_json_files_fast(
            os.path.join(PREDICTIONS_DIR, model_path),
            label_method=label_method,
            replays=replays,
            max_workers=max_workers,
            force=force,
        )
        annotation = get_annotations(prediction)
        topk_annotation = dataframe_topk_score(annotation, K=k)
        topk_annotation["idx"] = topk_annotation.groupby(
            ["replay", "image_id"]
        ).cumcount()  # 0~4 인덱스 생성
        wide_prediction = topk_annotation.pivot(
            index=["replay", "image_id"], columns="idx", values="bbox"
        )
        wide_prediction.columns = [f"pred_{i}" for i in wide_prediction.columns]
        wide_prediction.reset_index(inplace=True)
    
        result = pd.merge(
            ground_truth, wide_prediction, on=["replay", "image_id"], how="inner"
        )
        result = result.bfill()
        result = result.dropna()
    
        new_cols = []
        for c in result.columns:
            if c in ("replay", "image_id"):
                new_cols.append(("meta", c))
            elif c.startswith("gt_"):
                new_cols.append(("gt", c.split("_")[1]))
            elif c.startswith("pred_"):
                new_cols.append(("pred", c.split("_")[1]))
            else:
                new_cols.append(("other", c))  # fallback
        result.columns = pd.MultiIndex.from_tuples(new_cols)
    
        output = add_union_metrics_parallel(
            result,
            gt_cols=[("gt", str(i)) for i in range(5)],
            pred_cols=[("pred", str(i)) for i in range(k)],
            use_bbox_size=False,
            thresholds=(0.0, 0.3, 0.5),
        )
    
        gt_cols = result.columns[output.columns.get_level_values(0).str.startswith("union")]
        g = output.groupby(("meta", "replay"))[gt_cols].mean()
        out_per_replay = g.T.groupby(level=1).mean().T
        return output, out_per_replay

### SET 1

In [6]:
experiment_set1 = select_set("set1")

In [7]:
experiment_set1

ExperimentSet(name='set1', train=(36, 212, 438, 522, 1660, 1559, 1628, 2351, 6219, 11251), test=(275, 1725, 3613, 4520, 4664), models={'vanilla': PosixPath('all_correct_win4_b16_20250826_045543/model_020'), 'kbrs': PosixPath('all_correct_win4_b16_kbrs_20250912_005403/model_020')}, suffix='.rep')

In [8]:
ground_truth_set1 = get_ground_truth(replays=list(experiment_set1.test_replays))

In [9]:
output_vanilla_set1, out_per_replay_vanilla_set1 = get_inference_result(
    experiment_set1.model_vanilla(),
    ground_truth=ground_truth_set1,
    replays=list(experiment_set1.test_replays),
    k=1
)

In [10]:
output_kbrs_set1, out_per_replay_kbrs_set1 = get_inference_result(
    experiment_set1.model_kbrs(),
    ground_truth=ground_truth_set1,
    replays=list(experiment_set1.test_replays),
    k=1
)

In [11]:
out_per_replay_vanilla_set1.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.788299,0.761978,0.731245,1.471740,0.658682
std,0.035752,0.040652,0.045543,0.092697,0.042515
min,0.744920,0.710340,0.670706,1.376642,0.603228
25%,0.764820,0.739276,0.713974,1.389351,0.641403
50%,0.784066,0.755062,0.718423,1.466741,0.646352
75%,0.815628,0.796303,0.770992,1.532456,0.694646
max,0.832059,0.808908,0.782128,1.593511,0.707782


In [12]:
out_per_replay_kbrs_set1.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.821655,0.797918,0.765875,1.586748,0.703419
std,0.024874,0.025000,0.025078,0.052093,0.026078
min,0.785424,0.757292,0.723371,1.535650,0.660085
25%,0.805778,0.791043,0.762930,1.546902,0.701588
50%,0.837228,0.808171,0.779026,1.568472,0.708491
75%,0.839679,0.815270,0.780313,1.627984,0.721275
max,0.840166,0.817817,0.783734,1.654730,0.725658


### SET 1 PRIME

In [13]:
# experiment_set1_ = select_set("set1_prime")

In [14]:
# output_vanilla_set1_, out_per_replay_vanilla_set1_ = get_inference_result(
#     experiment_set1_.model_vanilla(),
#     ground_truth=ground_truth_set1,
#     replays=list(experiment_set1_.test_replays),
#     k=1
# )

In [15]:
# out_per_replay_vanilla_set1_.describe()

### SET 2

In [16]:
experiment_set2 = select_set("set2")

In [17]:
ground_truth_set2 = get_ground_truth(replays=list(experiment_set2.test_replays))

In [18]:
output_vanilla_set2, out_per_replay_vanilla_set2 = get_inference_result(
    experiment_set2.model_vanilla(),
    ground_truth=ground_truth_set2,
    replays=list(experiment_set2.test_replays),
    k=1
)

In [19]:
output_kbrs_set2, out_per_replay_kbrs_set2 = get_inference_result(
    experiment_set2.model_kbrs(),
    ground_truth=ground_truth_set2,
    replays=list(experiment_set2.test_replays),
    k=1
)

[FAIL] /workspace/predictions/all_correct_win4_b16_kbrs_20250915_071540/model_020/6219.rep/all_correct.json -> JSONDecodeError: unexpected end of data: line 41318158 column 13 (char 626318175)


### SET 3

In [20]:
experiment_set3 = select_set("set3")

In [21]:
ground_truth_set3 = get_ground_truth(replays=list(experiment_set3.test_replays))

In [24]:
output_vanilla_set3, out_per_replay_vanilla_set3 = get_inference_result(
    experiment_set3.model_vanilla(),
    ground_truth=ground_truth_set3,
    replays=list(experiment_set3.test_replays),
    k=1
)

ValueError: No objects to concatenate

In [ ]:
output_kbrs_set3, out_per_replay_kbrs_set3 = get_inference_result(
    experiment_set3.model_kbrs(),
    ground_truth=ground_truth_set3,
    replays=list(experiment_set3.test_replays),
    k=1
)

In [ ]:
# inference_legacy_set3 = get_inference(
#     experiment_set3.model_legacy(),
#     replays=list(experiment_set3.test_replays),
#     k=1
#     )

In [ ]:
# wide_inference = pd.concat([ground_truth_set3.set_index(['replay', 'image_id']), inference_legacy_set3.set_index(['replay', 'image_id'])], axis=1).reset_index()

# wide_inference.columns = pd.MultiIndex.from_tuples(
#     [
#         ('meta', 'replay'),
#         ('meta', 'image_id'),
#         ('gt', '0'),
#         ('gt', '1'),
#         ('gt', '2'),
#         ('gt', '3'),
#         ('gt', '4'),
#         ('pred', '0'),
#     ]
# )
# wide_inference = wide_inference.dropna()

In [ ]:
# output_legecy_set3 = add_union_metrics_parallel(wide_inference)

In [ ]:
# output_legecy_set3['union'].describe()[['ic@000', 'ic@030', 'ic@050', 'ic_multi', 'ic_ratio']]

In [ ]:
# output_legacy_set3, out_per_replay_legacy_set3 = get_inference_result(
#     experiment_set3.model_legacy(),
#     ground_truth=ground_truth_set3,
#     replays=list(experiment_set3.test_replays),
#     k=1
# )

In [16]:
out_per_replay_vanilla_set3.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.797899,0.770781,0.735577,1.512832,0.668827
std,0.036314,0.042351,0.048159,0.223883,0.051847
min,0.744947,0.705830,0.663740,1.174810,0.589981
25%,0.784754,0.755695,0.718443,1.436569,0.653216
50%,0.796278,0.778471,0.739921,1.528876,0.673710
75%,0.830601,0.801850,0.767324,1.691149,0.702290
max,0.832916,0.812058,0.788458,1.732754,0.724939


In [17]:
out_per_replay_kbrs_set3.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.824297,0.799788,0.773311,1.619885,0.707871
std,0.058849,0.066204,0.070564,0.258525,0.067996
min,0.728486,0.692626,0.662820,1.214054,0.599836
25%,0.809449,0.780428,0.743979,1.509275,0.683550
50%,0.851026,0.830550,0.807075,1.763241,0.741056
75%,0.855871,0.839230,0.819855,1.795034,0.747153
max,0.876654,0.856103,0.832825,1.817823,0.767759


In [ ]:
out_per_replay_legacy_set3.describe()

In [ ]:
# def compute_total_intersection_percent(out: pd.DataFrame):
#     if ("meta", "replay") not in out.columns:
#         if (
#             out.index.names
#             and "meta" in out.index.names
#             and "replay" in out.index.names
#         ):
#             out = out.reset_index()
#         else:
#             raise KeyError("('meta','replay')가 컬럼/인덱스 어디에도 없습니다.")

#     if ("union", "ic_ratio") not in out.columns:
#         raise KeyError(
#             "('union','ic_ratio') 컬럼이 없습니다. add_union_metrics_parallel 결과를 확인하세요."
#         )

#     overall_total = float(out[("union", "ic_ratio")].mean())

#     g = out.groupby(("meta", "replay"))[[("union", "ic_ratio")]].mean()
#     out_per_replay = g.T.groupby(level=1).mean().T
#     out_per_replay = out_per_replay.rename(
#         columns={"ic_ratio": "total_intersection_percent"}
#     )

#     return overall_total, out_per_replay

In [ ]:
# compute_total_intersection_percent(out)

In [ ]:
top5_annotation_prediction_kbrs = dataframe_topk_score(annotation_prediction_kbrs, K=5)

In [ ]:
top5_annotation_prediction_kbrs["idx"] = top5_annotation_prediction_kbrs.groupby(
    ["replay", "image_id"]
).cumcount()  # 0~4 인덱스 생성
wide_prediction_kbrs_top5 = top5_annotation_prediction_kbrs.pivot(
    index=["replay", "image_id"], columns="idx", values="bbox"
)
wide_prediction_kbrs_top5.columns = [
    f"pred_{i}" for i in wide_prediction_kbrs_top5.columns
]
wide_prediction_kbrs_top5.reset_index(inplace=True)

In [ ]:
result_inference_kbrs_top5 = pd.merge(
    ground_truth, wide_prediction_kbrs_top5, on=["replay", "image_id"], how="outer"
)
result_inference_kbrs_top5 = result_inference_kbrs_top5.bfill()
result_inference_kbrs_top5 = result_inference_kbrs_top5.dropna()

new_cols = []
for c in result_inference_kbrs_top5.columns:
    if c in ("replay", "image_id"):
        new_cols.append(("meta", c))
    elif c.startswith("gt_"):
        new_cols.append(("gt", c.split("_")[1]))
    elif c.startswith("pred_"):
        new_cols.append(("pred", c.split("_")[1]))
    else:
        new_cols.append(("other", c))  # fallback

result_inference_kbrs_top5.columns = pd.MultiIndex.from_tuples(new_cols)
result_inference_kbrs_top5.head()

In [ ]:
out_kbrs_top5 = add_union_metrics_parallel(
    result_inference_kbrs_top5,
    gt_cols=(("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols=(("pred", "0"), ("pred", "1"), ("pred", "2")),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5),
)

In [ ]:
out_kbrs_top5

In [ ]:
gt_cols = result_inference_kbrs_top5.columns[
    out_kbrs_top5.columns.get_level_values(0).str.startswith("union")
]
g_kbrs = out_kbrs_top5.groupby(("meta", "replay"))[gt_cols].mean()
out_per_replay_kbrs_top5 = g_kbrs.T.groupby(level=1).mean().T
out_per_replay_kbrs_top5.loc[
    ["275.rep", "1725.rep", "3613.rep", "4520.rep", "4664.rep"]
]

In [ ]:
import numpy as np
import pandas as pd


def add_iou_columns(
    df: pd.DataFrame,
    gt_key: str = "gt",
    pred_key: str = "pred",
    thresholds=(0.0, 0.3, 0.5),
    attach: bool = True,
):
    # gt/pred에 공통으로 존재하는 pair 키(예: 0..4)
    keys = [k for k in df[gt_key].columns if k in df[pred_key].columns]
    if not keys:
        raise ValueError("No overlapping ('gt', k) and ('pred', k) columns.")

    out = {}
    for k in keys:
        # 각 셀: [x, y, w, h] (top-left, width, height)
        gt = np.asarray(df[(gt_key, k)].to_list(), dtype=float)  # (N,4)
        pred = np.asarray(df[(pred_key, k)].to_list(), dtype=float)  # (N,4)

        xg, yg, wg, hg = gt.T
        xp, yp, wp, hp = pred.T

        # x2,y2
        xg2, yg2 = xg + wg, yg + hg
        xp2, yp2 = xp + wp, yp + hp

        # 교집합
        ix1 = np.maximum(xg, xp)
        iy1 = np.maximum(yg, yp)
        ix2 = np.minimum(xg2, xp2)
        iy2 = np.minimum(yg2, yp2)
        iw = np.clip(ix2 - ix1, 0, None)
        ih = np.clip(iy2 - iy1, 0, None)
        inter = iw * ih

        # 합집합
        area_g = np.clip(wg, 0, None) * np.clip(hg, 0, None)
        area_p = np.clip(wp, 0, None) * np.clip(hp, 0, None)
        union = area_g + area_p - inter

        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

        # IoU 값
        out[("iou", k, "value")] = iou

        # 임계값별 일치 여부
        for thr in thresholds:
            tag = f"iou@{int(round(thr*100)):03d}"  # 0.3 -> 'iou@030'
            out[("iou", k, tag)] = iou >= thr

    iou_df = pd.DataFrame(out)
    iou_df.columns = pd.MultiIndex.from_tuples(
        iou_df.columns, names=["metric", "pair", "comp"]
    )
    return pd.concat([df, iou_df], axis=1) if attach else iou_df

In [ ]:
# 사용 예
# df = ...  # 스크린샷과 같은 멀티인덱스 컬럼 DataFrame
df_iou = add_iou_columns(out_kbrs_top5, thresholds=(0.0, 0.3, 0.5))

In [ ]:
import numpy as np
import pandas as pd


def add_pairwise_iou(
    df: pd.DataFrame,
    gt_key: str = "gt",
    pred_key: str = "pred",
    ensure_gt=None,  # 예: range(5) -> gt 0..4 강제 생성
    ensure_pred=None,  # 예: range(6) -> pred 0..5 강제 생성
    thresholds=(0.0, 0.3, 0.5),  # 임계값 bool 컬럼도 생성(0.0은 inter>0)
    fill_invalid=np.nan,  # 유효치 없을 때 채울 값 (np.nan 권장)
    out_key: str = "iou",  # IoU 값 컬럼의 레벨0 이름
):
    # --- 서브키 수집 ---
    def _subs(top):
        if top in df.columns.get_level_values(0):
            return list(df[top].columns)
        return []

    def _kmap(subs):  # "표기 문자열" -> "원래 키"
        return {str(k): k for k in subs}

    gt_subs, pred_subs = _subs(gt_key), _subs(pred_key)
    gmap, pmap = _kmap(gt_subs), _kmap(pred_subs)

    def _sort_keys_str(keys):
        return sorted(keys, key=lambda s: (0, int(s)) if s.isdigit() else (1, s))

    if ensure_gt is not None:
        gt_names_str = [str(k) for k in ensure_gt]
    else:
        gt_names_str = _sort_keys_str(list(gmap.keys()))

    if ensure_pred is not None:
        pred_names_str = [str(k) for k in ensure_pred]
    else:
        pred_names_str = _sort_keys_str(list(pmap.keys()))

    N, G, P = len(df), len(gt_names_str), len(pred_names_str)

    # --- Series -> (N,4) 변환 ---
    def _to_xywh(series):
        if series is None:
            return np.full((N, 4), np.nan, dtype=float)
        try:
            arr = np.asarray(series.to_list(), dtype=float)
            if arr.ndim != 2 or arr.shape[1] < 4:
                raise ValueError
            return arr[:, :4]
        except Exception:
            arr = np.full((N, 4), np.nan, dtype=float)
            vals = series.to_list()
            for i, v in enumerate(vals):
                if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 4:
                    try:
                        arr[i, :4] = np.array(v[:4], dtype=float)
                    except Exception:
                        pass
            return arr

    # --- gt, pred 배열 구성: (N,G,4), (N,P,4) ---
    g = np.full((N, G, 4), np.nan, dtype=float)
    p = np.full((N, P, 4), np.nan, dtype=float)

    for gi, gs in enumerate(gt_names_str):
        s = df[(gt_key, gmap[gs])] if gs in gmap else None
        g[:, gi, :] = _to_xywh(s)

    for pj, ps in enumerate(pred_names_str):
        s = df[(pred_key, pmap[ps])] if ps in pmap else None
        p[:, pj, :] = _to_xywh(s)

    # 유효 여부 (각 박스 4요소 모두 수치)
    g_valid = ~np.isnan(g).any(axis=2)  # (N,G)
    p_valid = ~np.isnan(p).any(axis=2)  # (N,P)
    valid_pair = g_valid[:, :, None] & p_valid[:, None, :]  # (N,G,P)

    # --- 브로드캐스팅으로 (N,G,P) IoU 계산 ---
    wg = np.clip(g[:, :, 2], 0, None)[:, :, None]
    hg = np.clip(g[:, :, 3], 0, None)[:, :, None]
    wp = np.clip(p[:, :, 2], 0, None)[:, None, :]
    hp = np.clip(p[:, :, 3], 0, None)[:, None, :]

    xg1 = g[:, :, 0][:, :, None]
    yg1 = g[:, :, 1][:, :, None]
    xp1 = p[:, :, 0][:, None, :]
    yp1 = p[:, :, 1][:, None, :]
    xg2 = xg1 + wg
    yg2 = yg1 + hg
    xp2 = xp1 + wp
    yp2 = yp1 + hp

    ix1 = np.maximum(xg1, xp1)
    iy1 = np.maximum(yg1, yp1)
    ix2 = np.minimum(xg2, xp2)
    iy2 = np.minimum(yg2, yp2)

    iw = np.clip(ix2 - ix1, 0, None)
    ih = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    area_g = wg * hg
    area_p = wp * hp
    union = area_g + area_p - inter

    iou = np.full((N, G, P), fill_invalid, dtype=float)
    can = valid_pair & (union > 0)
    iou[can] = inter[can] / union[can]

    # --- 컬럼 생성 (2-레벨: metric, pair) ---
    out = {}
    for gi, gs in enumerate(gt_names_str):
        gt_lbl = int(gs) if gs.isdigit() else gs
        for pj, ps in enumerate(pred_names_str):
            pr_lbl = int(ps) if ps.isdigit() else ps
            pair_label = f"g{gt_lbl}_p{pr_lbl}"

            # IoU 실수값
            out[(out_key, pair_label)] = iou[:, gi, pj]

            # # 임계값 bool (0.0은 "조금이라도 겹치면": inter>0)
            # if thresholds is not None:
            #     for thr in thresholds:
            #         name = f"iou@{int(round(thr*100)):03d}"
            #         mask = np.zeros(N, dtype=bool)
            #         if thr == 0.0:
            #             cond = valid_pair[:, gi, pj] & (inter[:, gi, pj] > 0)
            #         else:
            #             cond = valid_pair[:, gi, pj] & (iou[:, gi, pj] >= thr)
            #         mask[cond] = True
            #         out[(name, pair_label)] = mask

    out_df = pd.DataFrame(out)
    out_df.columns = pd.MultiIndex.from_tuples(out_df.columns, names=["metric", "pair"])
    return pd.concat([df, out_df], axis=1)

In [ ]:
# 사용 예
# df = ...  # ('gt',0..4), ('pred',0..4) 형태의 멀티인덱스 컬럼 DataFrame
df2 = add_pairwise_iou(out_kbrs_top5)  # ('iou', k) 컬럼들이 추가됨
# pair0_iou = df2[("iou", 0)].head()  # pair=0의 IoU 값 확인

In [ ]:
df2[["meta", "iou"]]

In [ ]:
df2["iou"].loc[~df2["iou"].fillna(0).eq(0).all(axis=1)]

In [ ]:
len(df2["iou"].loc[~df2["iou"].fillna(0).eq(0).all(axis=1)]) / len(df2["iou"])

In [ ]:
import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0) 'iou' 멀티컬럼 -> (N,G,P) 큐브로 변환
#    df.columns level0='iou', level1='g{i}_p{j}' 형식 가정
# ------------------------------------------------------------
def _extract_iou_cube(df, iou_key="iou"):
    iou_df = df.xs(iou_key, axis=1, level=0)  # pair 라벨 컬럼만
    pair_cols = list(iou_df.columns)

    def _parse_pair(c):
        m = re.match(r"g(\d+)_p(\d+)", str(c))
        return (int(m.group(1)), int(m.group(2))) if m else None

    parsed = [_parse_pair(c) for c in pair_cols]
    g_ids = sorted({g for gp in parsed if gp for g in [gp[0]]})
    p_ids = sorted({p for gp in parsed if gp for p in [gp[1]]})
    g_index = {g: i for i, g in enumerate(g_ids)}
    p_index = {p: j for j, p in enumerate(p_ids)}

    N, G, P = len(iou_df), len(g_ids), len(p_ids)
    cube = np.full((N, G, P), np.nan, dtype=float)

    for col, gp in zip(pair_cols, parsed):
        if gp is None:
            continue
        gi, pj = g_index[gp[0]], p_index[gp[1]]
        cube[:, gi, pj] = iou_df[col].to_numpy(dtype=float)

    return cube, g_ids, p_ids


# ------------------------------------------------------------
# (A) Pair별 요약 테이블: mean IoU, hit rate@t, count
# ------------------------------------------------------------
def summarize_pairs(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    G, P = len(g_ids), len(p_ids)

    mean_iou = np.nanmean(cube, axis=0)  # (G,P)
    count = np.sum(~np.isnan(cube), axis=0)

    # t별 hit rate (t=0.0은 IoU>0)
    hit_rates = {}
    for t in thresholds:
        if t == 0.0:
            hits = cube > 0
        else:
            hits = cube >= t
        hit = np.nanmean(hits.astype(float), axis=0)  # 평균 = 비율
        hit_rates[t] = hit

    idx = [f"g{g}" for g in g_ids]
    cols = [f"p{p}" for p in p_ids]
    out = {
        "mean_iou": pd.DataFrame(mean_iou, index=idx, columns=cols),
        "count": pd.DataFrame(count, index=idx, columns=cols),
    }
    for t, mat in hit_rates.items():
        out[f"hit@{int(round(t*100)):03d}"] = pd.DataFrame(mat, index=idx, columns=cols)
    return out


# ------------------------------------------------------------
# (B) Best-of 매칭(중복 허용): GT Recall@t, Pred Precision@t, F1@t
#   - GT recall@t: 각 (n,g)에서 max_j IoU >= t
#   - Pred precision@t: 각 (n,p)에서 max_i IoU >= t
#   - 분모는 해당 행에서 "유효한 gt/pred"만 포함 (전부 NaN 제외)
# ------------------------------------------------------------
def bestof_metrics(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    N, G, P = cube.shape

    # 유효 gt/pred (해당 축에 NaN만 있으면 비유효)
    gt_present = ~np.isnan(cube).all(axis=2)  # (N,G)
    pred_present = ~np.isnan(cube).all(axis=1)  # (N,P)

    res = {}
    for t in thresholds:
        if t == 0.0:
            gt_hit = np.nanmax(cube, axis=2) > 0  # (N,G)
            pred_hit = np.nanmax(cube, axis=1) > 0  # (N,P)
        else:
            gt_hit = np.nanmax(cube, axis=2) >= t
            pred_hit = np.nanmax(cube, axis=1) >= t

        # 마스크로 분모/분자 집계 (NaN row 제외)
        gt_recall = (gt_hit & gt_present).sum() / np.maximum(gt_present.sum(), 1)
        pr_prec = (pred_hit & pred_present).sum() / np.maximum(pred_present.sum(), 1)
        f1 = 0.0
        if gt_recall + pr_prec > 0:
            f1 = 2 * gt_recall * pr_prec / (gt_recall + pr_prec)

        res[t] = dict(recall_gt=gt_recall, precision_pred=pr_prec, f1=f1)
    return res


# ------------------------------------------------------------
# (C) 일대일 매칭(Hungarian/그리디)
#   - 각 행 n에서 GT×Pred IoU 행렬에서 t 이상만 후보로 1:1 매칭
#   - 행마다 매칭된 페어의 개수/IoU를 합산하여 전체 Recall/Precision/F1, mIoU
#   - SciPy가 없으면 greedy(내림차순)로 대체
# ------------------------------------------------------------
try:
    from scipy.optimize import linear_sum_assignment
except Exception:
    linear_sum_assignment = None


def one2one_metrics(df, thresholds=(0.0, 0.3, 0.5)):
    cube, g_ids, p_ids = _extract_iou_cube(df)
    N, G, P = cube.shape
    metrics = {}

    for t in thresholds:
        matched = 0
        iou_sum = 0.0
        denom_gt = 0  # 전체 gt 유효 개수의 합
        denom_pr = 0  # 전체 pred 유효 개수의 합

        for n in range(N):
            M = cube[n]  # (G,P)
            # 유효한 gt/pred 만 고려
            g_mask = ~np.isnan(M).all(axis=1)
            p_mask = ~np.isnan(M).all(axis=0)
            Gi = np.where(g_mask)[0]
            Pj = np.where(p_mask)[0]

            denom_gt += len(Gi)
            denom_pr += len(Pj)
            if len(Gi) == 0 or len(Pj) == 0:
                continue

            subM = M[np.ix_(Gi, Pj)]  # (g', p')

            # 임계 미만은 후보 제외
            if t == 0.0:
                cand = subM > 0
            else:
                cand = subM >= t

            if not np.any(cand):
                continue

            if linear_sum_assignment is not None:
                # invalid에 큰 cost, 나머지는 cost=1-IoU
                cost = np.where(cand, 1.0 - np.clip(subM, 0, 1), 1e6)
                ri, cj = linear_sum_assignment(cost)
                for r, c in zip(ri, cj):
                    v = subM[r, c]
                    if (t == 0.0 and v > 0) or (t > 0.0 and v >= t):
                        matched += 1
                        iou_sum += float(v)
            else:
                # Greedy: IoU 내림차순으로 GT/Pred 중복 배제
                items = [
                    (i, j, subM[i, j])
                    for i in range(subM.shape[0])
                    for j in range(subM.shape[1])
                ]
                items = [
                    x
                    for x in items
                    if (t == 0.0 and x[2] > 0) or (t > 0.0 and x[2] >= t)
                ]
                items.sort(key=lambda x: x[2], reverse=True)
                used_r, used_c = set(), set()
                for r, c, v in items:
                    if r in used_r or c in used_c:
                        continue
                    used_r.add(r)
                    used_c.add(c)
                    matched += 1
                    iou_sum += float(v)

        recall = matched / max(denom_gt, 1)
        prec = matched / max(denom_pr, 1)
        f1 = 0.0
        if recall + prec > 0:
            f1 = 2 * recall * prec / (recall + prec)
        miou = (iou_sum / matched) if matched > 0 else 0.0
        metrics[t] = dict(
            recall_gt=recall, precision_pred=prec, f1=f1, mIoU=miou, matched=matched
        )

    return metrics

In [ ]:
pair_summ = summarize_pairs(df2, thresholds=(0.0, 0.3, 0.5))
pair_summ["mean_iou"]  # g×p 평균 IoU

In [ ]:
pair_summ["hit@000"]  # g×p hit rate@0.5

In [ ]:
pair_summ["count"]  # 비교에 사용된 샘플 수

In [ ]:
bo = bestof_metrics(df2, thresholds=(0.0, 0.3, 0.5))

In [ ]:
bo

In [ ]:
m11 = one2one_metrics(df2, thresholds=(0.0, 0.3, 0.5))

In [ ]:
m11